# Colab'da 2D PINN Tam Eğitimi (GPU)

Bu defter, 2D radyal dam-break PINN'ini **GPU ile tam eğitir** (CPU'da ağır olan kısım).

## Kullanım (3 adım)
1. **GPU'yu aç:** Üst menü → `Runtime` → `Change runtime type` → **T4 GPU** → Save.
2. Hücreleri **sırayla** çalıştır (Shift+Enter).
3. 3. hücre dosya isteyince bilgisayarından **`PINN-BAP-colab.zip`**'i seç.

> Tam 2D eğitim T4 GPU'da ~10-30 dk sürebilir. Daha hızlı deneme için 5. hücreden önce
> `config/param_ranges_2d.yaml` içinde `full.adam_iters`'i 8000-10000'e düşürebilirsin.

## 1) GPU kontrolü

In [ ]:
import torch
if torch.cuda.is_available():
    print('✓ GPU aktif:', torch.cuda.get_device_name(0))
else:
    print('⚠ GPU YOK! Runtime > Change runtime type > T4 GPU seç ve tekrar çalıştır.')

## 2) Bağımlılıklar (torch Colab'da hazır; sadece deepxde + pyyaml)

In [ ]:
!pip -q install deepxde pyyaml

## 3) Projeyi yükle (PINN-BAP-colab.zip seç)

In [ ]:
import zipfile, os, shutil
from google.colab import files

up = files.upload()                      # PINN-BAP-colab.zip seç
zips = [k for k in up if k.endswith('.zip')]
assert zips, ('Zip yüklenmedi. "Choose Files" ile PINN-BAP-colab.zip seç, '
              'yükleme bitene kadar BEKLE, sonra bu hücreyi tekrar çalıştır.')
with zipfile.ZipFile(zips[0]) as z:
    z.extractall('/content')

# Kökü otomatik bul — zip'te üst klasör olsa da olmasa da çalışır
ROOT = next((c for c in ['/content/PINN-BAP', '/content']
             if os.path.isdir(os.path.join(c, 'src', 'pinn'))), None)
assert ROOT, 'Proje bulunamadı — zip içeriğini kontrol et.'
if ROOT == '/content':                   # dosyalar /content'e dağıldıysa topla
    os.makedirs('/content/PINN-BAP', exist_ok=True)
    for item in ['src','config','validation','cases','notebooks','docs',
                 'requirements.txt','README.md']:
        if os.path.exists(f'/content/{item}'):
            shutil.move(f'/content/{item}', f'/content/PINN-BAP/{item}')
    ROOT = '/content/PINN-BAP'
os.chdir(ROOT)
print('✓ Proje kökü:', ROOT)
print('  içerik:', sorted(os.listdir(ROOT)))

In [ ]:
%cd /content/PINN-BAP

## 4) Hızlı kontrol (opsiyonel) — smoke ile boru hattı çalışıyor mu?

In [ ]:
!DDE_BACKEND=pytorch python src/pinn/pinn_baseline_2d.py --smoke

## 5) TAM eğitim (GPU) — asıl iş
30k Adam + L-BFGS. Birkaç on dakika sürebilir; çıktı `results/2d/`'e yazılır.

In [ ]:
!DDE_BACKEND=pytorch python src/pinn/pinn_baseline_2d.py

## 6) Doğrulama — heatmap + radyal kesit + L2 hatası

In [ ]:
!python validation/compare_baseline_2d.py
from IPython.display import Image, display
display(Image('results/2d/h_heatmap.png'))
display(Image('results/2d/radial_cut.png'))

## 7) Sonuçları indir (kopmadan önce!)
Colab oturumu kapanınca dosyalar silinir; sonuçları zip'leyip indir.

In [ ]:
import shutil
shutil.make_archive('/content/PINN-BAP-2d-results', 'zip', 'results/2d')
from google.colab import files
files.download('/content/PINN-BAP-2d-results.zip')

---
### Not: 1D deneysel modeli de GPU'da eğitmek istersen
```
!python src/data/parse_vosoughi2021.py
!DDE_BACKEND=pytorch python src/pinn/pinn_baseline.py --scenario experimental
!python validation/compare_experimental_pinn.py
```